# Flow Matching

**Flow matching** can be understood as the deterministic sibling of continuous-time diffusion modeling. In the diffusion chapter, a stochastic differential equation transported a simple distribution toward the data distribution, and the same marginals could also be realized through a probability flow ODE. Flow matching starts directly from that deterministic viewpoint. Instead of first defining a noisy stochastic process and then extracting an associated ODE, one defines a time-dependent probability path between an easy source distribution and the target data distribution, then learns a velocity field whose flow realizes that path.

This perspective is attractive for several reasons. It avoids explicit simulation of stochastic noise during sampling, it connects naturally with continuous normalizing flows, and it separates the design of the probability path from the learning of the transport dynamics. The central question is no longer how to reverse a noising process, but how to choose and learn a vector field that pushes mass in the right way over time.

For course purposes, flow matching is important because it reveals that the diffusion story was never only about denoising Gaussian noise. It was also about learning a path between a simple distribution and the data distribution. Flow matching keeps that path-based perspective but removes the stochastic wrapper. This makes the chapter a natural culmination of several earlier ideas: latent geometry from VAEs, transport thinking from diffusion, and regression-style training objectives that turn generative modeling into a learnable supervised problem.

The chapter is also pedagogically useful because it forces us to separate two questions that are often conflated. The first question is *which family of distributions should connect source and target over time?* The second is *which velocity field realizes that family?* Flow matching treats the first as a design choice and the second as a learning problem. That separation is one of the most conceptually elegant features of the method.

## Deterministic Transport and the Continuity Equation

Let $\boldsymbol{x}(t)$ be a deterministic trajectory solving the ODE $\frac{d\boldsymbol{x}}{dt} = \boldsymbol{v}(\boldsymbol{x}(t), t)$, where $\boldsymbol{v}$ is a time-dependent velocity field. If the initial condition $\boldsymbol{x}(0)$ is random with density $p_0$, then the density $p_t$ of $\boldsymbol{x}(t)$ evolves according to a conservation law. Intuitively, probability mass does not disappear. It is merely redistributed by the flow. This conservation principle leads to the continuity equation
:::{math}
\partial_t p_t(\boldsymbol{x})
+
\nabla_{\boldsymbol{x}} \cdot \big(p_t(\boldsymbol{x}) \boldsymbol{v}(\boldsymbol{x}, t)\big)
= 0.
:::
The equation says that temporal change in density is balanced by the divergence of probability flux.

```{prf:theorem} Continuity equation for deterministic flows
:label: thm-continuity-equation

Suppose $\boldsymbol{x}(t)$ solves the ODE
:::{math}
\frac{d\boldsymbol{x}}{dt} = \boldsymbol{v}(\boldsymbol{x}(t), t)
:::
and let $p_t$ denote the density of $\boldsymbol{x}(t)$. Under standard regularity assumptions, the family $(p_t)_{t \in [0,1]}$ satisfies
:::{math}
\partial_t p_t(\boldsymbol{x})
+
\nabla_{\boldsymbol{x}} \cdot \big(p_t(\boldsymbol{x}) \boldsymbol{v}(\boldsymbol{x}, t)\big)
= 0.
:::
```

```{prf:proof}
A full proof may be given in weak form. Let $\varphi$ be a smooth compactly supported test function. Then
:::{math}
\frac{d}{dt}\mathbb{E}[\varphi(\boldsymbol{x}(t))]
=
\mathbb{E}\left[
    \nabla \varphi(\boldsymbol{x}(t))^\top
    \boldsymbol{v}(\boldsymbol{x}(t), t)
\right]
:::
by the chain rule. Writing expectations as integrals against $p_t$ gives
:::{math}
\frac{d}{dt}\int \varphi(\boldsymbol{x}) p_t(\boldsymbol{x})\, d\boldsymbol{x}
=
\int \nabla \varphi(\boldsymbol{x})^\top
\boldsymbol{v}(\boldsymbol{x}, t)
p_t(\boldsymbol{x})\, d\boldsymbol{x}.
:::
Integrating by parts moves the gradient from $\varphi$ onto the probability flux, yielding
:::{math}
\int \varphi(\boldsymbol{x})
\left[
    \partial_t p_t(\boldsymbol{x})
    +
    \nabla_{\boldsymbol{x}} \cdot
    \big(p_t(\boldsymbol{x}) \boldsymbol{v}(\boldsymbol{x}, t)\big)
\right]
d\boldsymbol{x}
= 0.
:::
Since this holds for every test function $\varphi$, the quantity in brackets must vanish, which proves the claim.
```

This equation is the deterministic counterpart of the Fokker-Planck equation studied implicitly in the diffusion chapter. There, density evolution involved both drift and stochastic diffusion. Here only transport remains. Flow matching is therefore a method for learning transport fields rather than score-corrected stochastic dynamics.

It is helpful to interpret the continuity equation physically. Imagine colored dye being moved by a fluid velocity field. The dye concentration changes at each location not because mass is created or destroyed, but because the fluid carries it elsewhere. Probability density behaves in the same way under deterministic flow. This picture is often more intuitive than the partial differential equation itself, and it clarifies why vector-field design is central.

The analogy also helps explain why deterministic transport can be attractive computationally. If one can learn a vector field that moves mass coherently along a good path, one may avoid some of the random fluctuation handling that appears in diffusion-based sampling. The whole burden then shifts to choosing a path and matching the right transport velocity.

## Probability Paths

The next modeling ingredient is a probability path $(p_t)_{t \in [0,1]}$ such that $p_0$ is easy to sample and $p_1 = p_{gt}$ is the target data distribution. The source $p_0$ is often chosen to be a standard Gaussian. The target is the data distribution. Once a path is fixed, one wants a vector field $\boldsymbol{v}(\boldsymbol{x}, t)$ such that the continuity equation transports $p_0$ along that path and reaches $p_1$ at time one.

At first sight, this seems to require direct knowledge of the evolving density $p_t$, which is not available. The key simplification of flow matching is that one does not need to estimate the path density explicitly. Instead, one constructs a path through conditional interpolations between source and target samples and derives a corresponding conditional velocity field. Learning then becomes a regression problem toward that analytically known conditional target.

This is a decisive conceptual step. The hard object is the full path density $p_t$, which would normally be expensive to characterize. Flow matching avoids that difficulty by shifting attention to conditional paths given paired endpoint samples. Once those conditional paths are chosen carefully, their instantaneous velocities are explicit. The model is then trained on those explicit local targets rather than on an intractable density-level objective.

In this sense, flow matching continues a pattern we have already seen several times in the course. A generative problem that is difficult when formulated directly becomes manageable after introducing the right auxiliary structure. In VAEs that auxiliary structure was the variational distribution. In diffusion it was the noising process. In flow matching it is the conditional probability path.

## Conditional Probability Paths

Let $\boldsymbol{x}_0 \sim p_0$ and $\boldsymbol{x}_1 \sim p_{gt}$. A simple example of conditional interpolation is the linear path $\boldsymbol{x}_t = (1-t)\boldsymbol{x}_0 + t\boldsymbol{x}_1$. Conditionally on the endpoints, this path has instantaneous velocity $\frac{d\boldsymbol{x}_t}{dt} = \boldsymbol{x}_1 - \boldsymbol{x}_0$. More generally, one may choose richer Gaussian or optimal-transport-inspired interpolations, but the central idea is unchanged: define a tractable conditional path and compute its conditional velocity exactly.

A concrete picture is helpful. If the source sample lies near the origin and the target sample lies on one arm of a spiral-shaped dataset, the linear path simply draws a straight segment between them. Flow matching then learns how such many local straight-line suggestions should average into a coherent global transport field. This is why the method is richer than drawing one line between two points might first suggest.

```{admonition} Numerical Example: A Conditional Path in Two Dimensions
:class: numerical-example

Let the source point be $\boldsymbol{x}_0 = (0, 0)$ and the target point be $\boldsymbol{x}_1 = (4, 2)$. With the linear interpolation path $\boldsymbol{x}_t = (1-t)\boldsymbol{x}_0 + t\boldsymbol{x}_1$, the state at time $t = 0.25$ is $\boldsymbol{x}_{0.25} = (1, 0.5)$.

The conditional velocity is constant: $\boldsymbol{u}_t(\boldsymbol{x}_t | \boldsymbol{x}_0, \boldsymbol{x}_1) = \boldsymbol{x}_1 - \boldsymbol{x}_0 = (4, 2)$. So in this simple case the model is told that, at the point $(1, 0.5)$ and time $0.25$, the mass should keep moving in the direction $(4, 2)$. Training averages many such local instructions over many endpoint pairs until a global velocity field emerges.
```

```{prf:theorem} Conditional flow matching target
:label: thm-cfm-target

Let $(\boldsymbol{x}_0, \boldsymbol{x}_1)$ be sampled from a coupling between a source distribution $p_0$ and the target distribution $p_{gt}$. Let $\boldsymbol{x}_t$ be sampled from a conditional interpolation law $p_t(\boldsymbol{x} | \boldsymbol{x}_0, \boldsymbol{x}_1)$ with associated conditional velocity field $\boldsymbol{u}_t(\boldsymbol{x} | \boldsymbol{x}_0, \boldsymbol{x}_1)$. Then the minimizer of
:::{math}
\mathcal{L}_{CFM}(\theta)
=
\mathbb{E}
\left[
    \|
        \boldsymbol{v}_\theta(\boldsymbol{x}_t, t)
        -
        \boldsymbol{u}_t(\boldsymbol{x}_t | \boldsymbol{x}_0, \boldsymbol{x}_1)
    \|_2^2
\right]
:::
is the conditional expectation
:::{math}
\boldsymbol{v}_\theta^\star(\boldsymbol{x}, t)
=
\mathbb{E}
\big[
    \boldsymbol{u}_t(\boldsymbol{x}_t | \boldsymbol{x}_0, \boldsymbol{x}_1)
    \,\big|\,
    \boldsymbol{x}_t = \boldsymbol{x}
\big].
:::
```

```{prf:proof}
This is a standard orthogonality property of squared-error regression. For fixed $(\boldsymbol{x}, t)$, the best predictor of the random target
$\boldsymbol{u}_t(\boldsymbol{x}_t | \boldsymbol{x}_0, \boldsymbol{x}_1)$
given the input $(\boldsymbol{x}_t, t)$ is its conditional expectation given those inputs. Therefore the minimizer of the global mean-squared error is exactly the conditional expectation field displayed above.
```

The theorem is short, but it contains the practical power of flow matching. Instead of solving a hard density-matching problem directly, we turn learning into supervised regression against a conditional velocity that we can compute analytically from the chosen path. This is closely analogous to what happened in denoising diffusion: a difficult generative objective became a regression problem. The difference is that here the target is a velocity rather than a score or a noise vector.

There is an important subtlety here. The learned field is not merely the raw conditional velocity for one specific pair of endpoints. The regression optimum is the conditional expectation of that target given the current intermediate state and time. This is why a single global vector field can emerge from many different sampled endpoint pairs. The network learns the average transport behavior compatible with the chosen path family.

That observation also helps students see why flow matching is not trivial interpolation. The training data are generated from simple conditional constructions, but the resulting learned velocity field must generalize across the whole space and produce coherent global trajectories. The model is learning a transport law, not memorizing individual lines between points.

## Relation with Continuous Normalizing Flows

Continuous normalizing flows also learn ODE dynamics that transform a source distribution into a target distribution. The crucial distinction is that CNFs are usually trained by maximum likelihood, which requires repeated evaluation of divergence terms and numerical integration inside the loss. Flow matching avoids this expensive likelihood-based training objective by prescribing a path and matching its velocity field directly. Sampling remains ODE-based, but training becomes much more straightforward.

This is a major conceptual advantage for teaching. One can present flow matching as a deterministic generative model with neural ODE sampling, while keeping the learning rule as a simple regression problem. In that sense, it occupies an appealing middle ground: more directly transport-oriented than diffusion, but easier to train than classical likelihood-based CNFs.

Put differently, CNFs and flow matching may use very similar samplers at generation time while differing substantially in how the vector field is trained. CNFs typically ask the model to be simultaneously a transport map and a tractable likelihood model. Flow matching drops the likelihood burden during training and focuses directly on matching transport targets. This often makes the optimization story easier to explain and, in many settings, easier to execute.

## Relation with Diffusion and Probability Flow ODEs

The diffusion chapter already introduced the probability flow ODE associated with a stochastic forward process. Flow matching may be viewed as starting from the opposite end. Instead of deriving a deterministic transport equation from a stochastic model, it directly postulates a deterministic transport mechanism through a chosen probability path. This is why the two families are closely related. Both ultimately learn time-dependent vector fields that transport probability mass from a simple source to the data distribution. The main difference lies in how the vector field target is obtained.

In score-based diffusion, the field is built from the score of perturbed densities. In flow matching, the field is built from a conditional path construction. This difference can have significant computational consequences. Flow matching can permit larger integration steps at sample time and avoids score-specific stochastic machinery during training, while still keeping a clean continuous-time viewpoint.

This comparison is useful because it shows that diffusion and flow matching are not opponents so much as neighboring design choices in a shared space of transport-based generative models. Diffusion begins with a stochastic path and derives a usable vector field from score information. Flow matching begins with a deterministic path design and learns the corresponding field directly. The families therefore differ less in destination than in route.

For a PhD audience, this is a good moment to emphasize that modern generative modeling is increasingly about path design, solver choice, and field parameterization rather than about one monolithic notion of "the generator." This shift in viewpoint is part of what makes the recent literature both rich and conceptually unifying.

## Gaussian and Optimal-Transport-Inspired Paths

The linear interpolation path is conceptually simple, but it is not the only choice. One may define Gaussian bridges whose mean interpolates between source and target while the variance shrinks over time, or paths inspired by optimal transport couplings that encourage straighter, more efficient mass movement. The design of the path matters because it changes the regression target seen during training and therefore changes the geometry of the learned transport.

From a teaching perspective, this is an excellent moment to emphasize that a generative model is never only a neural network architecture. It is also a choice of latent geometry, path design, and objective. In VAEs this geometry was hidden in the latent variable and the ELBO. In diffusion it was encoded in the noising process. In flow matching it is encoded directly in the probability path.

The choice of path can have surprisingly concrete consequences. If the interpolation path is poorly aligned with the geometry of the data, the target velocity field may become unnecessarily curved or difficult to learn. If the path is closer to an optimal transport route, the learned trajectories may become straighter and numerically easier to integrate. This is why path design is not a decorative theoretical freedom. It is part of the effective inductive bias of the model.

In practice, this also means that flow matching opens a larger design space than one first sees from the basic linear interpolation example. The training objective is simple, but the family of useful paths is rich. For students, this is a valuable reminder that simplicity of the loss does not imply narrowness of the modeling framework.

## Rectified Flow Matching for Images

The implementation notebook will now move from the two-dimensional picture to images through **Rectified Flow Matching**. The idea is intentionally close to the conditional path already discussed. We sample a source image-shaped tensor $\boldsymbol{x}_0 \sim p_0$ from a simple noise distribution, usually a standard Gaussian on the same tensor space as the data, and we sample a data image $\boldsymbol{x}_1 \sim p_{gt}$. We then connect the two endpoints by the straight path
:::{math}
\boldsymbol{x}_t = (1-t)\boldsymbol{x}_0 + t\boldsymbol{x}_1,
\qquad t \in [0,1].
:::
For this path, the conditional velocity is again
:::{math}
\boldsymbol{u}_t = \frac{d\boldsymbol{x}_t}{dt} = \boldsymbol{x}_1 - \boldsymbol{x}_0.
:::
The adjective *rectified* emphasizes that the learned dynamics are encouraged to move samples along straightened transport trajectories from noise to data. In the simplest training setup, the endpoint coupling is independent: one draws a noise tensor and a data image independently, forms the interpolated tensor, and asks a neural network to predict the endpoint difference. This is not the only possible coupling, but it is the cleanest one for a first image-generation implementation.

```{prf:theorem} Rectified flow regression objective
:label: thm-rectified-flow-objective

Let $\boldsymbol{x}_0 \sim p_0$ and $\boldsymbol{x}_1 \sim p_{gt}$ be sampled from a chosen coupling, and define the linear path
:::{math}
\boldsymbol{x}_t = (1-t)\boldsymbol{x}_0 + t\boldsymbol{x}_1.
:::
For a time-conditioned vector field $\boldsymbol{v}_\theta(\boldsymbol{x},t)$, consider the objective
:::{math}
\mathcal{L}(\theta)
=
\mathbb{E}_{t,\boldsymbol{x}_0,\boldsymbol{x}_1}
\left[
\left\|
\boldsymbol{v}_\theta(\boldsymbol{x}_t,t) - (\boldsymbol{x}_1 - \boldsymbol{x}_0)
\right\|_2^2
\right],
:::
where $t \sim \mathcal{U}(0,1)$. For every fixed pair $(\boldsymbol{x},t)$, the pointwise minimizer is
:::{math}
\boldsymbol{v}^*(\boldsymbol{x},t)
=
\mathbb{E}\left[\boldsymbol{x}_1 - \boldsymbol{x}_0 | \boldsymbol{x}_t = \boldsymbol{x}\right].
:::
```

```{prf:proof}
The proof is the same conditional-expectation argument used for conditional flow matching, but it is useful to restate it in the image-generation notation. Fix the value of $(\boldsymbol{x}_t,t)$ and view the random vector $\boldsymbol{x}_1 - \boldsymbol{x}_0$ as the regression target. The expected squared error over this conditional distribution is minimized by the conditional mean. Therefore the best vector field at the point $(\boldsymbol{x},t)$ is the average endpoint velocity among all endpoint pairs whose straight interpolation passes through $\boldsymbol{x}$ at time $t$. This gives the displayed expression.
```

At generation time the learned field is used exactly as a deterministic transport model. One starts from $\boldsymbol{x}(0) \sim p_0$ and integrates
:::{math}
\frac{d\boldsymbol{x}}{dt} = \boldsymbol{v}_\theta(\boldsymbol{x}(t),t),
\qquad t \in [0,1].
:::
If the learned field is accurate and the numerical solver is sufficiently fine, the terminal sample $\boldsymbol{x}(1)$ should look like an image from $p_{gt}$. This is the point at which flow matching becomes directly comparable with the VAE, GAN, and diffusion implementations from the previous chapters: all four methods begin from simple randomness, use a neural network to shape that randomness, and produce image samples that can be inspected visually or evaluated with distribution-level metrics such as FID and KID.

The comparison with diffusion is especially instructive. A DDPM learns to reverse a stochastic noising process through many local denoising moves. Rectified flow learns a deterministic velocity field along a prescribed path from noise to data. Both objectives are regression losses with randomly sampled times. The targets differ: diffusion predicts noise or score-related quantities, while rectified flow predicts a transport velocity. This makes the image-scale implementation a useful final lab because it reuses many architectural ideas from diffusion, especially time embeddings and U-Net-like spatial processing, while changing the mathematical object that the network is asked to learn.

## Final Perspective for the Course

Flow matching completes the conceptual arc of the course. We began with latent-variable models that emphasized tractable probabilistic surrogates. We then studied adversarial training, where a learned critic replaces explicit likelihoods. We next moved to diffusion, where generation becomes denoising along a stochastic path. Flow matching shows that one can keep the continuous-time transport viewpoint while removing the stochastic component and learning a deterministic velocity field directly through regression.

The next notebook will implement a minimal flow matching example so that these ideas become concrete in code. The core modern reference is {cite}`lipman2023flow`, and the relation between transport design and more general couplings connects naturally to current work such as {cite}`tong2024improving`.

Conceptually, this final chapter should leave the reader with a sharpened perspective on what the course has really been about. Across VAEs, GANs, diffusion, and flow matching, we repeatedly designed a simple source of randomness, a parameterized mechanism for transforming or interpreting it, and an objective that made learning feasible. The languages changed, but the structural question remained the same: how should probability mass be organized so that realistic images become learnable and samplable?

Flow matching is therefore not just a recent add-on to the course. It is a particularly transparent endpoint of the larger narrative. Once students can see diffusion as path learning and flow matching as direct transport learning, the whole family of modern generative models becomes easier to compare at a conceptual level.